# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [14]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [15]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
    

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import os, json

class Document(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    inputTokens: int
    outputTokens: int

client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key= os.getenv('OPENAI_API_KEY'),
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

system_prompt = (
    "You are a summarisation assistant.  Given a block of text, you must "
    "reply with a *single* JSON object that conforms to the Document schema "
    "below.  Do not add any commentary or extraneous text.\n\n"
    "Schema: {\n"
    "  \"author\": \"<string>\",\n"
    "  \"title\": \"<string>\",\n"
    "  \"relevance\": \"<string>\",          # one paragraph, why an AI pro "
    "should care\n"
    "  \"summary\": \"<string>\",            # ≤1000 tokens\n"
    "  \"tone\": \"<string>\",               # e.g. Bureaucratese, Legalese, …\n"
    "  \"inputTokens\": <int>,\n"
    "  \"outputTokens\": <int>\n"
    "}\n"
    "The `summary` field must be written in a distinctive tone such as "
    "'Victorian English'"
)

user_prompt = f"Here is the document text:\n\n{document_text}"

response = client.responses.parse(
    model="gpt-4o-mini", 
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ],
    text_format=Document,
)

# the model’s raw text
model_output = response.output_parsed
print("raw output:\n", model_output)

# token counts are in the usage object
# in_tokens  = response.usage.prompt_tokens
# out_tokens = response.usage.completion_tokens

# parse into a Document instance
# doc = Document.model_validate_json(model_output)           # raises if JSON is invalid
# doc.inputTokens  = in_tokens
# doc.outputTokens = out_tokens

# print(doc.json(indent=2))

raw output:
 author='Peter F. Drucker' title='Managing Oneself' relevance="This article highlights the importance of self-management in today's evolving knowledge economy. AI professionals should care because it emphasizes understanding one's strengths and aligning personal values with career choices, principles that are crucial in any rapidly changing technological landscape. Navigating one's career with intentionality and self-awareness is essential for personal development and maximizing contributions in any field, including AI." summary='In this modern era of unprecedented opportunity, it is incumbent upon the individual to act as their own chief executive officer, crafting a meaningful career path. Mr. Drucker vehemently argues that knowledge workers must engage in meticulous introspection to discern their strengths, values, and optimal working styles. Through the noble practice of feedback analysis, individuals may unveil their latent proficiencies and frailties. The narrative ex

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel



# Result Data set as Key-Value Pair
class EvaluationReport(BaseModel):
    SummarizationScore: float = 0.0
    SummarizationReason: str = ""
    CoherenceScore: float = 0.0
    CoherenceReason: str = ""
    TonalityScore: float = 0.0
    TonalityReason: str = ""
    SafetyScore: float = 0.0
    SafetyReason: str = ""
  

report = EvaluationReport()

# Create OpenAI model for DeepEval 
model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# Create test case
test_case = LLMTestCase(input=document_text, actual_output=model_output.summary)

# Summarization Metric
summarizationMetric = SummarizationMetric(
    model=model,
    assessment_questions=[
        "Does the output perfectly match the Pydantic model structure?",
        "Is the summary written in a distinct Victorian English style?",
        "Does it explain why an AI pro should care in exactly one paragraph?", 
        "Are the input/output tokens actual numbers from the response object?",
        "Is the summary concise and strictly under 1,000 tokens?"
    ]
)

# Run summarization metric and update result to report
summarizationMetric.measure(test_case)
report.SummarizationScore = summarizationMetric.score
report.SummarizationReason = summarizationMetric.reason


# Clarity from Coherence
clarityMatric = GEval(
    model=model,
    name="Clarity",
    evaluation_steps=[
        "Does the 'actual output' follow a clear, natural sequence without abrupt jumps?",
        "Is the 'actual output' free from ambiguous pronouns or vague technical terms that require guesswork?",
        "Does the 'actual output' maintain a distinct beginning, middle, and end?",
        "Are complex AI concepts explained in a way that is easy to understand but technically accurate?",
        "Is the main message immediately clear to a professional reader on the first read?"
        ],
    evaluation_params=[ LLMTestCaseParams.ACTUAL_OUTPUT],
)

clarityMatric.measure(test_case)
report.CoherenceScore = clarityMatric.score
report.CoherenceReason = clarityMatric.reason

# Professionalism from Tonality
professionalismMatric = GEval(
    model=model,
    name="Professionalism",
    evaluation_steps=[
        "Check for accurate use of technical AI/ML terminology.",
        "Ensure the absence of slang, casual fillers, or overly 'chatty' AI phrases.",
        "Evaluate if the tone remains objective and evidence-based.",
        "Assess if the language is sophisticated enough for AI professional development.",
        "Verify if the Victorian style maintains a formal and respectful tone."
   ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

professionalismMatric.measure(test_case)
report.TonalityScore = professionalismMatric.score
report.TonalityReason = professionalismMatric.reason

# Diversity from Safety
diversityMatric = GEval(
    model=model,
    name="Diversity",
    evaluation_steps=[
        "Identify if the 'actual output'includes multiple perspectives or methodologies.",
        "Check for bias-free language regarding gender, race, or demographics.",
        "Evaluate if the 'actual output' covers technical, ethical, and business impacts.",
        "Ensure the Victorian style avoids historical stereotypes or exclusionary language.",
        "Assess if the relevance applies to a broad, global audience of AI professionals."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

diversityMatric.measure(test_case)
report.SafetyScore = diversityMatric.score
report.SafetyReason = diversityMatric.reason

print(report.model_dump_json(indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.0,
  "SummarizationReason": "The score is 0.00 because the summary contains significant contradictions to the original text, misrepresenting its focus and key concepts, which undermines its accuracy and reliability.",
  "CoherenceScore": 0.7232586709559011,
  "CoherenceReason": "The response follows a clear sequence and maintains a distinct beginning, middle, and end, effectively outlining the importance of self-management in career development. However, it contains some vague terms like 'noble practice' and 'latent proficiencies' that may require further clarification for a professional reader. While the main message is generally clear, the use of complex language could hinder immediate understanding for some readers.",
  "TonalityScore": 0.6352852205184888,
  "TonalityReason": "The response demonstrates a good use of technical terminology related to career development and self-management, aligning with the evaluation steps. However, it includes some casual

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [36]:
system_prompt_2 = "You are an editing expert."
user_prompt_2 = f"""There is a evaluation summary from previously generated summary. And, here is a result of it.
    Summarization Score: {report.SummarizationScore} (Reason: {report.SummarizationReason})
    Clarity Score (Coherence): {report.CoherenceScore} (Reason: {report.CoherenceReason})
    Professionalism Score (Tonality): {report.TonalityScore} (Reason: {report.TonalityReason})
    Diversity Score (Safety): {report.SafetyScore} (Reason: {report.SafetyReason})

    Improve and rewrite the summary by addressing these specific points based on this feedback.
    Here is the original document text:\n\n{document_text}
    """

response_2 = client.responses.parse(
    model="gpt-4o-mini", 
    input=[
        {"role": "system", "content": system_prompt_2},
        {"role": "user",   "content": user_prompt_2},
    ],
    text_format=Document,
)

# the model’s raw text
model_output_2 = response_2.output_parsed
print("raw output:\n", model_output_2)


report_2 = EvaluationReport()
test_case_2 = LLMTestCase(input=document_text, actual_output=model_output_2.summary)

summarizationMetric.measure(test_case_2)
report_2.SummarizationScore = summarizationMetric.score
report_2.SummarizationReason = summarizationMetric.reason

clarityMatric.measure(test_case_2)
report_2.CoherenceScore = clarityMatric.score
report_2.CoherenceReason = clarityMatric.reason

professionalismMatric.measure(test_case_2)
report_2.TonalityScore = professionalismMatric.score
report_2.TonalityReason = professionalismMatric.reason

diversityMatric.measure(test_case_2)
report_2.SafetyScore = diversityMatric.score
report_2.SafetyReason = diversityMatric.reason


print("--- First ----\n")
print(report.model_dump_json(indent=2))
print("\n\n")
print("--- Second ----\n")
print(report_2.model_dump_json(indent=2))



## Report my output. Did I get a better result? Do you think thses control are enough?
## I got improved results in coherence, tonality and safety evaluations after self-correction.
## However, it still returned 0 evaluation score for summarization evaluation of second output.
## I observe the limitation of current control approach, especially based on the summarization evaluation result.
## According to the reasons provided by DeepEval, the first output generated distorted content, 
## and the second output, in an attempt to correct that distortion, "imagined" information not found in the original text 


Output()

raw output:
 author='Peter F. Drucker' title='Managing Oneself' relevance='Self-management in the knowledge economy' summary='In the modern knowledge economy, success comes to those who deeply understand themselves, including their strengths, weaknesses, values, and preferred work styles. Instead of relying on organizations to advance their careers, individuals must take on the responsibility of being their own chief executive officers. To achieve true excellence, one must identify their strengths through methods like feedback analysis, understand their work styles—whether they function better in groups or solo—and align their values with organizational missions to ensure satisfaction and effectiveness. Lastly, individuals should consider their optimal work environments and what contributions they can uniquely make to their organizations, ultimately leading to a fulfilling career over a potentially long working life.' tone='Professional, Objective, and Reflective on Workforce Dynamics'

Output()

Output()

Output()

--- First ----

{
  "SummarizationScore": 0.0,
  "SummarizationReason": "The score is 0.00 because the summary contains significant contradictions to the original text, misrepresenting its focus and key concepts, which undermines its accuracy and reliability.",
  "CoherenceScore": 0.7232586709559011,
  "CoherenceReason": "The response follows a clear sequence and maintains a distinct beginning, middle, and end, effectively outlining the importance of self-management in career development. However, it contains some vague terms like 'noble practice' and 'latent proficiencies' that may require further clarification for a professional reader. While the main message is generally clear, the use of complex language could hinder immediate understanding for some readers.",
  "TonalityScore": 0.6352852205184888,
  "TonalityReason": "The response demonstrates a good use of technical terminology related to career development and self-management, aligning with the evaluation steps. However, it incl

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
